In [1]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import pandas as pd
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
import pickle
import ast
from time import time

BASE_DIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"

# remote 
#NEO4J_URL ="bolt://31.207.47.254:7687"
#NEO4J_USER = "neo4j"
#NEO4J_PWD = "password"

# local
NEO4J_URL ="bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PWD = "neo4j"

import sys 
sys.path.insert(0, "../")

from src.neo4j_functions import Neo4jConnection

In [2]:
class MyEmbeddingFunction(EmbeddingFunction):
    def __init__(self, embedder):
        self.embedder = embedder
    def __call__(self, input: Documents) -> Embeddings:
        return self.embedder.embed_documents(input)

In [3]:
# !!! BELOW TO CHANGE !!! 
DATA_NAME = 'vectorized_triplets'
DB_VERSION = 'v1'
GRAPH_DB_NAME = 'neo4j'


EMBEDDING_MODEL_PATH = f'{BASE_DIR}/models/facebook/contriever'
MODEL_KWARGS = {'device': 'cuda'}
ENCODE_KWARGS = {'normalize_embeddings': True}
CHROMA_KWARGS = {"hnsw:space": "ip"}
# !!! ABOVE TO CHANGE !!!

SAVE_DIR = f"../data/{DATA_NAME}/{DB_VERSION}"
DENSE_DB_SAVE_PATH = f'{SAVE_DIR}/densedb'
DB_LOG_PATH = f'{SAVE_DIR}/operation_info.json' 

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_PATH,
    model_kwargs=MODEL_KWARGS,
    encode_kwargs=ENCODE_KWARGS 
)
ef = MyEmbeddingFunction(embeddings)

No sentence-transformers model found with name /home/dzigen/Desktop/PersonalAI/Personal-AI/models/facebook/contriever. Creating a new one with MEAN pooling.


In [5]:
client = chromadb.PersistentClient(path=DENSE_DB_SAVE_PATH)
collection = client.get_or_create_collection(name=DATA_NAME,  metadata=CHROMA_KWARGS, 
                                             embedding_function=ef)

In [6]:
conn = Neo4jConnection(uri=NEO4J_URL, user=NEO4J_USER, pwd=NEO4J_PWD)

In [7]:
output = conn.execute_query("MATCH (a)-[rel]-(b) RETURN rel", db=GRAPH_DB_NAME)

In [ ]:
# TO MODIFY 
# - how to stringify triplets?

graph_entities = list(map(lambda item: (item['n'].element_id, item['n']['name']), output))

documents = list(map(lambda item: item[1], graph_entities))
metadata = list(map(lambda item: {'node_id': item[0]}, graph_entities))

In [ ]:
conn.close()

#### Vectorizing 

In [ ]:
vectorize_t_start = time()

collection.add(
    documents=documents,
    metadatas=metadata,
    ids=list(map(lambda v: v['triplet_id'], metadata))
)

VECTORIZE_ELAPSED_TIME = round(time() - vectorize_t_start, 5)

In [ ]:
collection.count()

#### Saving Log

In [ ]:
with open(DB_LOG_PATH, 'w') as fd:
    fd.write(json.dumps({
        "data_name": DATA_NAME, "graphdb_name": GRAPH_DB_NAME,
        "db_version": DB_VERSION, "model_name": EMBEDDING_MODEL_PATH,
        "encode_kwargs": ENCODE_KWARGS, "chroma_kwargs": CHROMA_KWARGS,
        "vectorize_elapsed_sec_time": VECTORIZE_ELAPSED_TIME}, indent=1))